# 11.9 - Retrieval

**Phase:** 11 - RAG Systems

**Status:** VERIFIED

---

## 1. What Are We Solving?

Retrieval finds the most relevant chunks for a user query using vector search. This is the core of RAG - if the right chunk isn't retrieved, the LLM answers from irrelevant context. Retrieval quality is the *ceiling* of RAG quality.

## 2. Why Does This Matter?

You have an index (11.8). Now query it: embed the query, search top-k, filter by metadata, and measure recall@k against known-good answers.

## 3. Prerequisites

Units 11.4, 11.5, 11.8.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Query Chroma and FAISS with an embedded query (top-k)
- Sweep k (1/3/5) and filter by metadata
- Compute recall@k and precision@k over a hand-built test set

## 5. Mental Model

Retrieval is a librarian fetching the most relevant books: quality depends on how well the librarian understands the question and how well the library is organized.

```text
User Query -> Embed Query -> Search Vector DB -> Return Top-k Chunks
```


## 6. Rebuild the FAQ Index
Reuse the 11.8 pattern: chunk a small corpus, embed (fallback-safe), store in Chroma.

In [1]:
# Deterministic embedding helper.
# Loads all-MiniLM-L6-v2 if available; otherwise falls back to a hash-based
# vector so every cell still completes offline. The fallback still gives
# "similar text -> similar vector" behaviour via character-bigram overlap,
# so the demos remain meaningful without the model download.
import hashlib, numpy as np

_DIM = 384


def _hash_embed(texts):
    vecs = np.zeros((len(texts), _DIM))
    for i, t in enumerate(texts):
        bigrams = [t[j:j+2].lower() for j in range(len(t)-1)]
        for bg in bigrams:
            h = int(hashlib.md5(bg.encode()).hexdigest(), 16) % _DIM
            vecs[i, h] += 1.0
        norm = np.linalg.norm(vecs[i]) or 1.0
        vecs[i] = vecs[i] / norm
    return vecs


_model = None
_model_name = "all-MiniLM-L6-v2"


def get_embedder(force_fallback=False):
    """Return a function texts -> np.ndarray (N, dim)."""
    global _model
    if force_fallback:
        return _hash_embed
    if _model is None:
        try:
            from sentence_transformers import SentenceTransformer
            _model = SentenceTransformer(_model_name)
        except Exception as e:
            print("MiniLM unavailable, using hash fallback:", type(e).__name__)
            _model = None
    if _model is None:
        return _hash_embed
    return lambda texts: np.asarray(_model.encode(list(texts), convert_to_numpy=True))


def embed(texts, force_fallback=False):
    fn = get_embedder(force_fallback=force_fallback)
    return np.asarray(fn(texts), dtype=np.float32)


print("embedding dim:", _DIM)
print("backend:", _model_name if get_embedder() != _hash_embed else "hash-fallback")


embedding dim: 384


D:\CODE\complete ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:  65%|██████▌   | 67/103 [00:00<00:00, 637.16it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 664.23it/s]

backend: all-MiniLM-L6-v2


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = {
    "d_returns": "You can return items within 30 days of purchase. Items must be unopened and in "
                 "original packaging. Shipping costs are non-refundable.",
    "d_shipping": "Standard shipping takes 5-7 business days. Express shipping costs extra and "
                  "arrives in 2-3 business days.",
    "d_refunds": "Refunds are issued to the original payment method within 5-7 business days of "
                 "receiving the returned item.",
    "d_warranty": "The warranty covers manufacturing defects for one year. Damage from misuse is "
                  "not covered.",
}
splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40,
                                          separators=["\n\n", "\n", ". ", " ", ""])
chunk_texts, chunk_ids, metas = [], [], []
for doc_id, txt in CORPUS.items():
    pieces = splitter.split_text(txt)
    for j, p in enumerate(pieces):
        chunk_texts.append(p)
        chunk_ids.append(f"{doc_id}__{j}")
        metas.append({"doc": doc_id})

import chromadb
client = chromadb.Client()
col = client.create_collection("retrieval_idx", embedding_function=None,
                               metadata={"hnsw:space": "cosine"})
emb = embed(chunk_texts)
col.add(documents=chunk_texts, embeddings=emb.tolist(), ids=chunk_ids, metadatas=metas)
print("indexed", col.count(), "chunks")


indexed 4 chunks


## 7. Query Embedding + Top-k from Chroma
The query must go through the **same** embedding that produced the index. We query top-3 and print source doc, distance, and text.

In [3]:
query = "Can I send back a product I bought?"
q_emb = embed([query]).tolist()
res = col.query(query_embeddings=q_emb, n_results=3)
for doc, dist, meta, cid in zip(res["documents"][0], res["distances"][0],
                                res["metadatas"][0], res["ids"][0]):
    print(f"dist={dist:.4f} doc={meta['doc']} id={cid}")
    print(f"   {doc[:70]}")


dist=0.4275 doc=d_returns id=d_returns__0
   You can return items within 30 days of purchase. Items must be unopene
dist=0.6516 doc=d_refunds id=d_refunds__0
   Refunds are issued to the original payment method within 5-7 business 
dist=0.8373 doc=d_warranty id=d_warranty__0
   The warranty covers manufacturing defects for one year. Damage from mi


## 8. FAISS Top-k
Same idea in FAISS: embed the query with the same helper and search the in-memory index.

In [4]:
import faiss

fidx = faiss.IndexFlatIP(emb.shape[1])
fidx.add(np.asarray(emb, dtype="float32"))
D, I = fidx.search(embed([query]).astype("float32"), 3)
print("faiss top-3 positions:", I[0])
for pos in I[0]:
    cid, txt, meta = chunk_ids[pos], chunk_texts[pos], metas[pos]
    print(f"  score={D[0][int(np.where(I[0]==pos)[0][0])]:.4f} doc={meta['doc']} | {txt[:50]}")


faiss top-3 positions: [0 2 3]
  score=0.5725 doc=d_returns | You can return items within 30 days of purchase. I
  score=0.3484 doc=d_refunds | Refunds are issued to the original payment method 
  score=0.1627 doc=d_warranty | The warranty covers manufacturing defects for one 


## 9. k Sweep + Metadata Filter
Larger k raises recall but adds noise. Filtering by metadata (e.g. only 'refund' docs) narrows the search space before ranking.

In [5]:
for k in (1, 3, 5):
    r = col.query(query_embeddings=q_emb, n_results=k)
    docs = {m["doc"] for m in r["metadatas"][0]}
    print(f"k={k}: top docs = {sorted(docs)}")

print("\nfiltered by doc == 'd_refunds':")
r = col.query(query_embeddings=q_emb, n_results=5, where={"doc": "d_refunds"})
print("  returned:", [m["doc"] for m in r["metadatas"][0]], "| text:", r["documents"][0][0][:40])


k=1: top docs = ['d_returns']
k=3: top docs = ['d_refunds', 'd_returns', 'd_warranty']
k=5: top docs = ['d_refunds', 'd_returns', 'd_shipping', 'd_warranty']

filtered by doc == 'd_refunds':
  returned: ['d_refunds'] | text: Refunds are issued to the original payme


## 10. Recall@k / Precision@k on a Hand-Built Test Set
We define queries with the doc ids that should be retrieved (gold), then compute recall@k and precision@k. This is the same evaluation you'd scale up to hundreds of queries.

In [6]:
def recall_precision(query, gold_ids, k):
    r = col.query(query_embeddings=embed([query]).tolist(), n_results=k)
    retrieved = {m["doc"] for m in r["metadatas"][0]}
    hits = retrieved & set(gold_ids)
    return len(hits) / len(gold_ids), len(hits) / k, sorted(retrieved)


test_set = [
    ("how do i return an item", ["d_returns"], 5),
    ("how fast is standard shipping", ["d_shipping"], 5),
    ("when will my refund post", ["d_refunds"], 5),
    ("does warranty cover breaking it myself", ["d_warranty"], 5),
]
print(f"{'query':38s} {'recall@5':>9s} {'prec@5':>8s} retrieved")
for q, gold, k in test_set:
    rec, prec, ret = recall_precision(q, gold, k)
    print(f"{q[:38]:38s} {rec:9.2f} {prec:8.2f} {ret}")


query                                   recall@5   prec@5 retrieved


how do i return an item                     1.00     0.20 ['d_refunds', 'd_returns', 'd_shipping', 'd_warranty']


how fast is standard shipping               1.00     0.20 ['d_refunds', 'd_returns', 'd_shipping', 'd_warranty']


when will my refund post                    1.00     0.20 ['d_refunds', 'd_returns', 'd_shipping', 'd_warranty']
does warranty cover breaking it myself      1.00     0.20 ['d_refunds', 'd_returns', 'd_shipping', 'd_warranty']



## Common Mistakes

- Using a different embedding model for queries than for indexing.
- Retrieving too few chunks (low recall) or too many (noise).
- Not filtering by metadata.
- Ignoring retrieval latency.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| Relevant doc not retrieved | k too small / wrong model | Increase k, verify model consistency |
| Too many irrelevant results | k too large / weak model | Reduce k, add reranking |
| Query slow | Large index, no proper index | Build ANN index |
| Empty metadata filter | Filter values don't match | Print metadata values |

## Best Practices

- Use the same embedding model for indexing and querying.
- Start with k=5, tune on evaluation.
- Log retrieved chunks with scores.
- Combine with metadata filtering when applicable.
- Evaluate retrieval before evaluating generation.

## Hands-On Practice

1. **Basic:** Query the 11.8 index with 3 different queries.
2. **Guided:** Vary k (1, 3, 5, 10) and observe the changes.
3. **Independent:** Add metadata filtering to vector search.
4. **Realistic:** Build a 20-query test set and measure recall@5.
5. **Challenge:** Implement hybrid search combining BM25 and vector results.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
